In [ ]:
#Imports
import sys
sys.path.append("../") #Path to seq_scan_summary.py
import seq_scan_summary
from seq_scan_summary import *

In [ ]:
#Main

if __name__ == "__main__":
    
    #Parameters

    systems = ["LC3B_FUNDC1", "LC3B_FUNDC1-STphosmim"]
    cand_fastas = {"WT": "fasta_files/FUNDC1.fasta", "ST": "fasta_files/FUNDC1-STphosmim.fasta"}
    fragment_lengths = [16, 36, 52]
    fragment_types = ["WT_16", "ST_16", "WT_36", "ST_36", "WT_52", "ST_52"]
    #WT: wild-type
    #ST: phosphomimetic sequence
    
    files={"WT_16": "./analysis_output/sp_Q9GZQ8_MLP3B_HUMAN_and_sp_Q8IVP5_FUND1_HUMAN_16mers_peaks.csv",
           "WT_36": "./analysis_output/sp_Q9GZQ8_MLP3B_HUMAN_and_sp_Q8IVP5_FUND1_HUMAN_36mers_peaks.csv",
           "WT_52": "./analysis_output/sp_Q9GZQ8_MLP3B_HUMAN_and_sp_Q8IVP5_FUND1_HUMAN_52mers_peaks.csv",
           "ST_16": "./analysis_output/sp_Q9GZQ8_MLP3B_HUMAN_and_sp_Q8IVP5_FUND1_HUMAN-STphosmim_16mers_peaks.csv",
           "ST_36": "./analysis_output/sp_Q9GZQ8_MLP3B_HUMAN_and_sp_Q8IVP5_FUND1_HUMAN-STphosmim_36mers_peaks.csv",
           "ST_52": "./analysis_output/sp_Q9GZQ8_MLP3B_HUMAN_and_sp_Q8IVP5_FUND1_HUMAN-STphosmim_52mers_peaks.csv"
          }
    
    names={"WT_16": "sp_Q9GZQ8_MLP3B_HUMAN_and_sp_Q8IVP5_FUND1_HUMAN",
           "WT_36": "sp_Q9GZQ8_MLP3B_HUMAN_and_sp_Q8IVP5_FUND1_HUMAN",
           "WT_52": "sp_Q9GZQ8_MLP3B_HUMAN_and_sp_Q8IVP5_FUND1_HUMAN",
           "ST_16": "sp_Q9GZQ8_MLP3B_HUMAN_and_sp_Q8IVP5_FUND1_HUMAN",
           "ST_36": "sp_Q9GZQ8_MLP3B_HUMAN_and_sp_Q8IVP5_FUND1_HUMAN",
           "ST_52": "sp_Q9GZQ8_MLP3B_HUMAN_and_sp_Q8IVP5_FUND1_HUMAN"
          }
    
    name_adds={"WT_16": ("", False),
               "WT_36": ("", False),
               "WT_52": ("", False),
               "ST_16": ("-STphosmim", True),
               "ST_36": ("-STphosmim", True),
               "ST_52": ("-STphosmim", True)
              }
    
    fragment_lengths = {"WT_16": 16,
                        "WT_36": 36,
                        "WT_52": 52,
                        "ST_16": 16,
                        "ST_36": 36,
                        "ST_52": 52
                       }

    fragment_dirs={"WT_16": "./empty_fragment_directories/alphapulldown_output/",
                   "WT_36": "./empty_fragment_directories/alphapulldown_output/",
                   "WT_52": "./empty_fragment_directories/alphapulldown_output/",
                   "ST_16": "./empty_fragment_directories/alphapulldown_output/",
                   "ST_36": "./empty_fragment_directories/alphapulldown_output/",
                   "ST_52": "./empty_fragment_directories/alphapulldown_output/"
                  } #Normally this would link to the proper alphapulldown output. In this example all files are removed from within the subdirectories to save space
    #Run
    
    peak_list=[] #Peak list for scoring and summary
    peak_list_for_WT_ST=[] #Peak list to evaluate impactful modifications
    
    for frag_type in files.keys():
        peak_list=peak_list+read_features_from_csv(files[frag_type], frag_type)
        peak_list_for_WT_ST=peak_list_for_WT_ST+read_features_from_csv(files[frag_type], frag_type, keep_original_frag=True)
    
    #Impact of modifcations
    get_closest_phosmim(input_peak_list=peak_list_for_WT_ST, frag_types=fragment_types, frag_names=names, frag_name_adds=name_adds, frag_dirs=fragment_dirs, frag_lens=fragment_lengths, 
                        mod_types=["WT", "ST"], mod_fastas=cand_fastas, mod_seq_starts_at_res={"WT": 1, "ST": 1}, min_motif_len=3, output_name=systems[0])
    plot_mod_dist(files=["LC3B_FUNDC1_mod_with_effect_dist_to_motif.csv"], output="output")
    
    #Summary of peaks
    uni_peaks=summarize_peaks(peak_list, fragment_types)
    up_uni_peaks=add_relative_occurences(unique_peaks=uni_peaks, frag_types=fragment_types, frag_names=names, frag_name_adds=name_adds, frag_dirs=fragment_dirs, frag_lens=fragment_lengths)
    plot_peak_summary(up_uni_peaks, fragment_types, "_".join(systems)+"_summary.png", use_n_first_peaks=5, sort_by="self", sorters=[(4,6),(2,4),(0,2)])
    
    #Computing and plotting summarizig scores
    compute_scores(up_uni_peaks)
    add_res_depth(up_uni_peaks, "./FUNDC1_AF2_res_depth.csv")
    write_unique_peaks_to_csv(up_uni_peaks, f"{str('_'.join(systems))}_unique_peaks_scored", overwrite=True)
    reloaded_peaks=load_unique_peaks_from_csv(f"{str('_'.join(systems))}_unique_peaks_scored.csv")
    plot_summary_scores([f"{str(systems[0])}"], [reloaded_peaks], ["blue"], output_name=f"{str('_'.join(systems))}_unique_peaks_scored")
    plot_peak_score(unique_peaks=reloaded_peaks, mod_types=["WT", "ST"], colors=["blue", "red"], output_name=systems[0], use_n_first_peaks=5, 
                    score=("length-weighted fraction", 0.0, 1.0), sort_by="all", plotwidth=6, use_peak_types=[], exclude_peak_types=["low-conf-LIR", "other@LDS", "other"], limit_av_depth_to=0.3)